<a href="https://colab.research.google.com/github/Patricia-oliv/Processamento-de-Linguagem-Natural-NLP-/blob/main/C%C3%B3pia_de_Entra21_Chatbot_Qwen_Contexto_Cardapio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Entra21 — Chatbot com Qwen e contexto externo

Neste notebook vamos acrescentar uma nova capacidade ao chatbot desenvolvido anteriormente: **usar informações de um arquivo de texto como contexto para responder ao usuário**.

Usaremos um cardápio fictício de uma lanchonete. O arquivo contém:

- lanches;
- ingredientes;
- preços;
- acompanhamentos;
- bebidas;
- regras de entrega;
- retirada no local;
- formas de pagamento;
- informações gerais.

## Objetivos

Ao final, você deverá compreender como:

- carregar um arquivo `.txt` no Google Colab;
- transformar o conteúdo do arquivo em uma string Python;
- inserir esse texto no contexto de um modelo de linguagem;
- combinar contexto externo com memória da conversa;
- medir quantos tokens estamos enviando ao modelo;
- construir um chatbot que responda com base em uma fonte externa.

> Nesta etapa ainda enviaremos **o documento inteiro** ao modelo.  
> Isso é uma forma simples de *context injection* ou *grounding*, mas ainda não é um RAG completo com recuperação de trechos.

## 1. Preparação do ambiente

Este notebook foi pensado para execução no **Google Colab com GPU**.

No Colab, selecione:

**Ambiente de execução → Alterar o tipo de ambiente de execução → GPU**

Vamos instalar as bibliotecas necessárias.

In [ ]:
!pip uninstall -y torchvision
!pip install -q -U transformers accelerate bitsandbytes

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 14.2 MB/s eta 0:00:00


### Verificando a GPU

O Qwen2.5-7B-Instruct será carregado com quantização em 4 bits. Mesmo assim, recomendamos usar uma GPU no Colab.

In [ ]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Atenção: este notebook foi planejado para execução com GPU.")

PyTorch: 2.11.0+cu128
Transformers: 5.17.0
CUDA disponível: True
GPU: Tesla T4


## 2. Carregando o Qwen2.5-7B-Instruct

Vamos usar a mesma configuração do notebook anterior:

- `AutoTokenizer`;
- `AutoModelForCausalLM`;
- quantização em 4 bits com `BitsAndBytesConfig`.

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

In [ ]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

print("Modelo carregado com sucesso.")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Modelo carregado com sucesso.


## 3. Qual é o tamanho da janela de contexto?

A janela de contexto corresponde à quantidade máxima de tokens que o modelo consegue considerar em uma chamada.

Podemos consultar a configuração carregada do próprio modelo.

In [ ]:
limite_contexto = getattr(
    model.config,
    "max_position_embeddings",
    None
)

print("Limite configurado de contexto:", limite_contexto)

Limite configurado de contexto: 32768


Esse limite precisa acomodar tudo o que enviamos ao modelo:

```text
mensagem de sistema
+
cardápio
+
histórico da conversa
+
pergunta atual
+
espaço para a resposta
```

Por isso, mesmo quando o documento cabe no contexto, é útil medir seu tamanho.

# Parte 2 — Carregando o cardápio

## 4. Fazendo upload do arquivo

Vamos carregar o arquivo:

`cardapio_byte_burger.txt`

No Colab, `files.upload()` abre uma janela para selecionar um arquivo do computador.

In [ ]:
from google.colab import files

arquivos_enviados = files.upload()

Saving 01M2JFEX3YM4Z96W7D11R20YV2.txt to 01M2JFEX3YM4Z96W7D11R20YV2.txt


## 5. Lendo o arquivo como texto

O upload retorna os arquivos em bytes.

Vamos selecionar o primeiro arquivo enviado e convertê-lo para uma string usando UTF-8.

In [ ]:
nome_arquivo = next(iter(arquivos_enviados))

contexto_cardapio = arquivos_enviados[nome_arquivo].decode("utf-8")

print("Arquivo carregado:", nome_arquivo)
print("Caracteres:", len(contexto_cardapio))

Arquivo carregado: 01M2JFEX3YM4Z96W7D11R20YV2.txt
Caracteres: 5209


### Inspecionando o conteúdo

Antes de enviar uma fonte externa ao modelo, é importante verificar se ela foi carregada corretamente.

Mostraremos apenas os primeiros 1.500 caracteres.

In [ ]:
print(contexto_cardapio[:1500])

LANCHONETE BYTE BURGER
Cardápio fictício para atividade didática de chatbot e RAG

LANCHES

1. BYTE BACON
Descrição:
Hambúrguer artesanal com bastante bacon crocante, queijo derretido e molho especial da casa.

Ingredientes:
Pão brioche, hambúrguer bovino de 150 g, bacon, queijo muçarela, alface, tomate, cebola roxa e molho especial.

Preço:
R$ 29,90


2. STACK DUPLO
Descrição:
Para quem está com muita fome: dois hambúrgueres artesanais, queijo em dobro e cebola caramelizada.

Ingredientes:
Pão brioche, dois hambúrgueres bovinos de 150 g, queijo cheddar, cebola caramelizada, picles e maionese defumada.

Preço:
R$ 37,90


3. CHICKEN CRUNCH
Descrição:
Sanduíche de frango empanado crocante com salada fresca e molho levemente picante.

Ingredientes:
Pão brioche, filé de frango empanado, queijo muçarela, alface, tomate, cebola roxa e molho de pimenta suave.

Preço:
R$ 27,90


4. VEGGIE CODE
Descrição:
Opção vegetariana com hambúrguer de grão-de-bico, vegetais frescos e molho de ervas.

Ingr

## 6. Quantos tokens existem no cardápio?

Modelos de linguagem não trabalham diretamente com palavras ou caracteres. O texto é convertido em **tokens**.

Vamos medir quantos tokens o cardápio ocupa usando o próprio tokenizador do Qwen.

In [ ]:
def contar_tokens_texto(texto):

    tokens = tokenizer(
        texto,
        add_special_tokens=False
    )

    return len(tokens.input_ids)


tokens_cardapio = contar_tokens_texto(contexto_cardapio)

print("Tokens do cardápio:", tokens_cardapio)

Tokens do cardápio: 1608


# Parte 3 — Injetando o cardápio no contexto

## 7. Criando a mensagem de sistema

No chatbot anterior, a mensagem de sistema dizia apenas como o assistente deveria se comportar.

Agora vamos acrescentar uma nova informação: **o conteúdo integral do cardápio**.

A estrutura será aproximadamente:

```text
INSTRUÇÕES
+
CARDÁPIO
+
HISTÓRICO DA CONVERSA
```

Também vamos orientar o modelo a não inventar informações que não estejam no documento.

In [ ]:
PROMPT_SISTEMA = f"""
Você é o atendente virtual da Lanchonete Byte Burger.

Sua tarefa é responder perguntas sobre produtos, ingredientes, preços,
bebidas, entrega, retirada e outras informações da lanchonete.

Utilize prioritariamente as informações presentes no CARDÁPIO abaixo.

Se uma informação não estiver disponível no cardápio, diga claramente
que essa informação não está disponível. Não invente preços, produtos,
ingredientes, horários ou regras.

Responda em português brasileiro, de forma clara e objetiva.

========================
CARDÁPIO
========================

{contexto_cardapio}

========================
FIM DO CARDÁPIO
========================
""".strip()

print("Prompt de sistema criado.")

Prompt de sistema criado.


## 8. O que aconteceu?

O cardápio não foi usado para treinar o Qwen.

Ele foi simplesmente inserido na mensagem de sistema.

Portanto, em cada nova chamada, o modelo receberá novamente:

```text
instruções
+
cardápio
+
conversa recente
+
nova pergunta
```

Essa é uma forma simples de fornecer **conhecimento externo temporário** ao modelo.

## 9. Criando a memória da conversa

Vamos reaproveitar a mesma lógica do notebook anterior.

A primeira mensagem do histórico será o `PROMPT_SISTEMA`, que agora já contém o cardápio.

In [ ]:
historico = [
    {
        "role": "system",
        "content": PROMPT_SISTEMA
    }
]

historico

[{'role': 'system',
  'content': 'Você é o atendente virtual da Lanchonete Byte Burger.\n\nSua tarefa é responder perguntas sobre produtos, ingredientes, preços,\nbebidas, entrega, retirada e outras informações da lanchonete.\n\nUtilize prioritariamente as informações presentes no CARDÁPIO abaixo.\n\nSe uma informação não estiver disponível no cardápio, diga claramente\nque essa informação não está disponível. Não invente preços, produtos,\ningredientes, horários ou regras.\n\nResponda em português brasileiro, de forma clara e objetiva.\n\n========================\nCARDÁPIO\n========================\n\nLANCHONETE BYTE BURGER\nCardápio fictício para atividade didática de chatbot e RAG\n\n==================================================\nLANCHES\n==================================================\n\n1. BYTE BACON\nDescrição:\nHambúrguer artesanal com bastante bacon crocante, queijo derretido e molho especial da casa.\n\nIngredientes:\nPão brioche, hambúrguer bovino de 150 g, bacon, que

# Parte 4 — Funções do chatbot

## 10. Função de geração

A função abaixo recebe uma lista de mensagens, aplica o *chat template* do Qwen e gera a resposta.

Ela não controla a memória. Sua única responsabilidade é conversar com o modelo.

In [ ]:
def gerar_resposta(messages, max_tokens=300):

    texto = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        texto,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    novos_tokens = outputs[0][inputs.input_ids.shape[1]:]

    resposta = tokenizer.decode(
        novos_tokens,
        skip_special_tokens=True
    )

    return resposta.strip()

## 11. Memória recente

Continuaremos armazenando o histórico completo, mas enviaremos ao modelo apenas os turnos recentes.

A mensagem de sistema — e, portanto, o cardápio — será sempre preservada.

In [ ]:
def montar_contexto(max_turnos=4):

    sistema = historico[0]
    conversa = historico[1:]

    # Pergunta atual + até N-1 pares anteriores
    max_mensagens_recentes = 2 * max_turnos - 1

    recentes = conversa[-max_mensagens_recentes:]

    return [sistema] + recentes

## 12. Função `conversar()`

A cada nova interação:

1. guardamos a pergunta;
2. montamos o contexto;
3. enviamos o contexto ao Qwen;
4. guardamos a resposta.

O cardápio permanece dentro da mensagem de sistema.

In [ ]:
def conversar(mensagem, max_tokens=300, max_turnos=4):

    historico.append(
        {
            "role": "user",
            "content": mensagem
        }
    )

    contexto = montar_contexto(
        max_turnos=max_turnos
    )

    resposta = gerar_resposta(
        contexto,
        max_tokens=max_tokens
    )

    historico.append(
        {
            "role": "assistant",
            "content": resposta
        }
    )

    return resposta

## 13. Limpando a conversa

Quando apagamos a memória, queremos apagar apenas a conversa.

O cardápio deve continuar disponível. Por isso, preservamos o `PROMPT_SISTEMA`.

In [ ]:
def limpar_memoria():

    historico.clear()

    historico.append(
        {
            "role": "system",
            "content": PROMPT_SISTEMA
        }
    )

    print("Memória da conversa apagada.")

## 14. Inspecionando a memória

Para não imprimir todo o cardápio cada vez que visualizarmos o histórico, mostraremos apenas uma indicação quando a mensagem for do tipo `system`.

In [ ]:
def mostrar_historico():

    for numero, mensagem in enumerate(historico):

        papel = mensagem["role"].upper()

        print(f"[{numero}] {papel}")

        if mensagem["role"] == "system":
            print("[Mensagem de sistema contendo instruções + cardápio]")
        else:
            print(mensagem["content"])

        print("-" * 60)

# Parte 5 — Medindo o contexto completo

## 15. Contando tokens das mensagens

Agora vamos contar tokens depois que o *chat template* foi aplicado.

Isso representa melhor o tamanho real da entrada enviada ao modelo.

In [ ]:
def contar_tokens_contexto(messages):

    texto = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    tokens = tokenizer(
        texto,
        add_special_tokens=False
    )

    return len(tokens.input_ids)

In [ ]:
contexto_atual = montar_contexto(max_turnos=4)

tokens_contexto = contar_tokens_contexto(contexto_atual)

print("Tokens do cardápio:", tokens_cardapio)
print("Tokens do contexto atual:", tokens_contexto)

if limite_contexto is not None:
    print("Limite configurado do modelo:", limite_contexto)
    print(
        "Percentual ocupado:",
        round(tokens_contexto / limite_contexto * 100, 2),
        "%"
    )

Tokens do cardápio: 1608
Tokens do contexto atual: 1760
Limite configurado do modelo: 32768
Percentual ocupado: 5.37 %


# Parte 6 — Testando o chatbot

## 16. Pergunta sobre um produto

Primeiro faremos uma pergunta cuja resposta está explicitamente no cardápio.

In [ ]:
print(
    conversar(
        "Quais lanches do cardápio têm bacon?"
    )
)

O BYTE BACON tem bacon como ingrediente.


## 17. Pergunta que combina diferentes informações

Agora a resposta exige consultar uma regra de entrega.

In [ ]:
print(
    conversar(
        "Se eu estiver a 5 km da lanchonete, quanto pago de taxa de entrega?"
    )
)

Para uma entrega a 5 km da lanchonete, a taxa de entrega é de R$ 8,00.


## 18. Pergunta sobre ingredientes e preferência

O modelo também pode combinar a pergunta do usuário com os ingredientes descritos no cardápio.

In [ ]:
print(
    conversar(
        "Sou vegetariano. Qual lanche do cardápio é mais adequado para mim?"
    )
)

O Veggie Code é uma opção vegetariana e adequada para você. Ele não contém carne e é feito com hambúrguer de grão-de-bico, vegetais frescos e um molho de ervas.


## 19. Pergunta sobre informação inexistente

Este teste é importante.

Vamos perguntar algo que não aparece no cardápio.

O comportamento desejado é admitir que a informação não está disponível, em vez de inventar uma resposta.

In [ ]:
print(
    conversar(
        "A lanchonete oferece pizza de calabresa?"
    )
)

Não, o cardápio da Byte Burger não inclui pizza de calabresa. Os lanches disponíveis são BYTE BACON, STACK DUPLO, CHICKEN CRUNCH e Veggie Code.


## 20. Testando memória + cardápio

Além de consultar o documento, o chatbot continua mantendo memória local da conversa.

Vamos fornecer uma preferência do usuário.

In [ ]:
print(
    conversar(
        "Eu não gosto de tomate. Lembre disso durante nossa conversa."
    )
)

Claro! Durante a preparação do seu pedido, podemos retirar o tomate sem custo adicional. Qual lanche você gostaria de pedir?


In [ ]:
print(
    conversar(
        "Então, qual lanche vegetariano você me recomendaria e que alteração eu deveria pedir?"
    )
)

Eu recomendo o Veggie Code, que é um lanche vegetariano. Você pode pedi-lo sem tomate, caso não goste desse ingrediente. Assim, os ingredientes serão:

- Pão integral
- Hambúrguer de grão-de-bico
- Queijo muçarela
- Alface
- Cebola roxa
- Cenoura ralada
- Molho de ervas

Esse lanche será preparado conforme suas preferências.


Observe que a resposta pode combinar duas fontes de contexto:

```text
CARDÁPIO
   +
MEMÓRIA DA CONVERSA
   ↓
QWEN
   ↓
RESPOSTA
```

O cardápio informa os ingredientes.

A conversa informa que o usuário não gosta de tomate.

# Parte 7 — Chatbot interativo

## 21. Reiniciando a conversa

Vamos apagar os testes antes de iniciar o modo interativo.

In [ ]:
limpar_memoria()

Memória da conversa apagada.


## 22. Interface simples

Comandos:

- `/sair` — encerra o chatbot;
- `/limpar` — apaga somente a memória da conversa;
- `/memoria` — mostra as mensagens armazenadas;
- `/tokens` — mostra quantos tokens existem no contexto atual.

In [ ]:
print("Byte Burger — Chatbot iniciado.")
print("Comandos: /sair | /limpar | /memoria | /tokens")

while True:

    mensagem = input("\nVocê: ").strip()

    if mensagem.lower() == "/sair":
        print("Conversa encerrada.")
        break

    if mensagem.lower() == "/limpar":
        limpar_memoria()
        continue

    if mensagem.lower() == "/memoria":
        mostrar_historico()
        continue

    if mensagem.lower() == "/tokens":
        contexto = montar_contexto(max_turnos=4)
        print(
            "Tokens no contexto:",
            contar_tokens_contexto(contexto)
        )
        continue

    if not mensagem:
        continue

    resposta = conversar(
        mensagem,
        max_tokens=300,
        max_turnos=4
    )

    print("\nByte Burger:")
    print(resposta)

Byte Burger — Chatbot iniciado.
Comandos: /sair | /limpar | /memoria | /tokens

Você: quero um lanche vegetariano

Byte Burger:
Peço desculpas pela confusão, mas o Veggie Code é a única opção vegetariana no nosso cardápio. Ele não é vegano porque contém queijo muçarela e o molho pode conter derivados de leite.

Você: lanche vegano

Byte Burger:
Desculpe, mas no momento não temos uma opção totalmente vegana no cardápio. O Veggie Code contém queijo muçarela e o molho pode conter derivados de leite, então não é vegano.


# O que construímos?

A arquitetura agora é:

```text
cardapio.txt
     │
     ▼
leitura do arquivo
     │
     ▼
string Python
     │
     ▼
mensagem de sistema
     │
     ├──────────────┐
     │              │
     │        memória recente
     │              │
     └───────┬──────┘
             ▼
            Qwen
             │
             ▼
          resposta
```

Temos agora um chatbot com:

- um modelo de linguagem;
- memória local;
- uma fonte externa de conhecimento;
- regras para responder com base nessa fonte.

# Isto já é RAG?

**Ainda não completamente.**

Neste notebook, enviamos o **cardápio inteiro** em todas as chamadas.

Não existe uma etapa que procure quais partes do documento são mais relevantes para cada pergunta.

Por exemplo, para responder:

> "Quanto custa o Byte Bacon?"

o modelo recebe também:

- cervejas;
- regras de retirada;
- formas de pagamento;
- horários;
- todos os outros produtos.

Na próxima etapa, criaremos uma fase de **recuperação**:

```text
PERGUNTA
   │
   ▼
buscar trechos relevantes no cardápio
   │
   ▼
selecionar apenas os melhores trechos
   │
   ▼
enviar os trechos ao Qwen
   │
   ▼
RESPOSTA
```

Essa arquitetura será nossa primeira implementação de **Retrieval-Augmented Generation (RAG)**.

## Atividade

Teste o chatbot com diferentes tipos de perguntas.

### Perguntas diretas

- Qual é o preço do Stack Duplo?
- Quais cervejas estão disponíveis?
- Qual é o pedido mínimo para entrega?

### Perguntas que exigem combinação de informações

- Quero um lanche sem carne bovina. Quais opções tenho?
- Estou a 4 km da lanchonete. Qual será a taxa?
- Quero retirar um lanche às 23h15. É possível?

### Perguntas com memória

Primeiro diga:

> Eu não como cebola.

Depois pergunte:

> Qual lanche você me recomenda e o que devo retirar?

### Perguntas sem resposta no documento

- Vocês têm sobremesa?
- Qual é o telefone da lanchonete?
- Há estacionamento?

Observe se o chatbot admite quando a informação não está disponível.